# 🚀 Step 2 — Run Huihui Qwen3.8 27B on Kaggle GPU

Launch **Huihui-Qwen3.8-27B-abliterated-UD-DW-Q4_K_M.gguf** with llama.cpp and expose its OpenAI-compatible API through a Cloudflare tunnel.

- **Model:** [huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF](https://huggingface.co/huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF), ~16.55 GB (15.42 GiB).
- **Vision projector:** `mmproj-model-bf16.gguf` from the same repository, ~0.93 GB. Use this matching projector for image input.
- **Flow:** client → Cloudflare → `localhost:8080` → `llama-server`.

### Prerequisites

- [ ] **Accelerator → GPU T4 x2** recommended. A single T4 needs partial CPU offloading and is slower.
- [ ] **Internet → ON** for Cloudflare and missing model downloads.
- [ ] Attach **`llama-cpp-bin`**, rebuilt with a **current llama.cpp** using `1b_setup_llama.ipynb`. The model author recommends the latest llama.cpp; older binaries may not support this GGUF or automatic GPU fitting.
- [ ] Optional: attach a dataset containing the exact model and projector filenames above. Otherwise Cell 1 downloads them from Hugging Face to `/kaggle/working/huihui-qwen38-27b`.

Optional: run `1_setup_dataset.ipynb` to create or update **your own private** `huihui-qwen38-27b-gguf` dataset with these exact files, then attach its newest version. Otherwise this notebook downloads missing files directly from Hugging Face. No prebuilt public model dataset is supplied by this repository.


## ⚙️ Cell 1 — Configuration and model files

All settings are here. Set `KAGGLE_USERNAME` if needed to select your datasets; it can stay empty when the attached files are unambiguous.

The model and matching projector are resolved independently: attached files first, then a resumable Hugging Face download. The revision is pinned so both downloads come from the same model repository snapshot. Allow ~17.5 GB plus download-cache overhead.

Use the new alias with the existing CLI client:
```bash
python chat.py --url https://xxxx.trycloudflare.com --model huihui-qwen3.8-27b-abliterated
python chat.py --url https://xxxx.trycloudflare.com --model huihui-qwen3.8-27b-abliterated --thinking
```

Thinking is selected per request with `chat_template_kwargs.enable_thinking`. Client sampling parameters override the server defaults.


In [ ]:
# CONFIGURATION — Edit this cell only
KAGGLE_USERNAME = ""  # Optional; otherwise use the environment or auto-detect inputs

# Optional attached model dataset; exact filenames are also searched under /kaggle/input
DATASET_NAME = "huihui-qwen38-27b-gguf"
MODEL_REPO = "huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF"
MODEL_REVISION = "c4b90228246aef7ecc6cb2be9d80575a911a0f3a"
MODEL_FILE = "Huihui-Qwen3.8-27B-abliterated-UD-DW-Q4_K_M.gguf"
MMPROJ_FILE = "mmproj-model-bf16.gguf"
ENABLE_VISION = True  # False: text only; do not load/download a projector
DOWNLOAD_IF_MISSING = True
MODEL_ALIAS = "huihui-qwen3.8-27b-abliterated"

# Rebuild with current llama.cpp using 1b_setup_llama.ipynb
LLAMA_DATASET = "llama-cpp-bin"

PORT = 8080
CTX_SIZE = 16384  # Reduce to 8192 if memory is tight
# Starting defaults; tune for your workload (not model-specific recommendations).
TEMPERATURE = 0.7
TOP_P = 0.8
TOP_K = 20
MIN_P = 0.0
PRESENCE_PENALTY = 1.5

# Current llama.cpp can fit layers to available VRAM with "auto".
# With a single T4, use "auto" or an explicit smaller layer count.
# 999 requests all layers; it does NOT guarantee an automatic CPU fallback.
N_GPU_LAYERS = "auto"

import os
import subprocess
import sys
from pathlib import Path

KAGGLE_USERNAME = KAGGLE_USERNAME or os.environ.get("KAGGLE_USERNAME", "")
INPUT_ROOT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)
DOWNLOAD_DIR = WORKING / "huihui-qwen38-27b"
LLAMA_SERVER = WORKING / "llama-server"


def find_input_file(filename, dataset_name):
    # Support both current owner/dataset mounts and older slug-only mounts.
    preferred_roots = [
        INPUT_ROOT / "datasets" / KAGGLE_USERNAME / dataset_name,
        INPUT_ROOT / dataset_name,
    ]
    for root in preferred_roots:
        candidate = root / filename
        if candidate.is_file():
            return candidate
    matches = sorted(p for p in INPUT_ROOT.rglob(filename) if p.is_file())
    if len(matches) > 1:
        raise RuntimeError(
            f"Multiple inputs contain {filename}: {matches}. "
            "Set KAGGLE_USERNAME and the dataset name, or detach duplicate inputs."
        )
    return matches[0] if matches else None


# Validate binaries before downloading large weights.
binary_input = find_input_file("llama-server", LLAMA_DATASET)
if binary_input is None:
    raise FileNotFoundError(
        "Attach a current llama-cpp-bin dataset. "
        "Re-run 1b_setup_llama.ipynb to build it with current llama.cpp."
    )
LLAMA_BIN_DIR = binary_input.parent


def resolve_model_file(filename):
    path = find_input_file(filename, DATASET_NAME)
    if path is None:
        if not DOWNLOAD_IF_MISSING:
            raise FileNotFoundError(
                f"Attach {filename} as a Kaggle input or set DOWNLOAD_IF_MISSING=True."
            )
        try:
            from huggingface_hub import hf_hub_download
        except ImportError:
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"],
                check=True,
            )
            from huggingface_hub import hf_hub_download
        print(f"📥 Resolving {filename} from {MODEL_REPO}...", flush=True)
        path = Path(hf_hub_download(
            repo_id=MODEL_REPO,
            filename=filename,
            revision=MODEL_REVISION,
            local_dir=str(DOWNLOAD_DIR),
        ))
    with path.open("rb") as f:
        if f.read(4) != b"GGUF":
            raise ValueError(f"Not a valid GGUF file: {path}")
    return path


MODEL_PATH = resolve_model_file(MODEL_FILE)
MMPROJ_PATH = resolve_model_file(MMPROJ_FILE) if ENABLE_VISION else None

print("✅ Configuration validated")
print(f"   Model        : {MODEL_PATH}")
print(f"   API alias    : {MODEL_ALIAS}")
print(f"   mmproj       : {MMPROJ_PATH or 'disabled (text only)'}")
print(f"   llama-cpp    : {LLAMA_BIN_DIR}")
print(f"   Context size : {CTX_SIZE} tokens")


## 🖥️ Cell 2 — GPU check

In [ ]:
import subprocess

result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"],
    capture_output=True, text=True
)

if result.returncode != 0:
    print("❌ No GPU detected. Go to Settings → Accelerator and select T4 GPU.")
else:
    lines = result.stdout.strip().split("\n")
    total_vram = 0
    print(f"{'GPU':<5} {'Name':<20} {'Total VRAM':>12} {'Free VRAM':>12}")
    print("-" * 55)
    for i, line in enumerate(lines):
        name, total, free = [x.strip() for x in line.split(",")]
        total_mb = int(total.replace(" MiB", ""))
        free_mb  = int(free.replace(" MiB", ""))
        total_vram += total_mb
        print(f"  {i:<3} {name:<20} {total_mb/1024:>10.1f} GB {free_mb/1024:>10.1f} GB")
    print("-" * 55)
    print(f"  Total available VRAM: {total_vram/1024:.1f} GB")
    print()
    model_size_gb = MODEL_PATH.stat().st_size / (1024**3)
    if total_vram / 1024 >= model_size_gb:
        print(f"✅ Weights: {model_size_gb:.1f} GiB; total GPU memory: {total_vram/1024:.1f} GiB.")
        print("   KV cache, compute buffers and vision need additional VRAM.")
    else:
        print(f"⚠️  VRAM ({total_vram/1024:.1f} GB) < model size ({model_size_gb:.1f} GB).")
        print("   Use N_GPU_LAYERS='auto' or reduce the layer count for CPU offloading.")
        print(f"   → Recommended: switch to 2x T4 in Kaggle accelerator settings.")

## 🔧 Cell 2b — Install CUDA 12 runtime dependencies

Install `nvidia-cuda-runtime-cu12` and `nvidia-cublas-cu12` with the notebook's Python to supply `libcudart.so.12` and `libcublas.so.12`. This step must run **before Cell 3**, whose `llama-server --version` check already needs these libraries.

The cell locates NVIDIA libraries in system and user site-packages, preserves existing driver/library paths, then checks loading in a fresh subprocess. Installation, missing-file or loader errors stop execution. These are user-space runtime libraries; the Kaggle GPU driver still comes from the host. Internet is required when packages are not already installed.


In [ ]:
# Install CUDA 12 runtime dependencies before checking or launching llama-server.
# Fix missing libcudart.so.12 / libcublas.so.12.
import os
import sys
import site
import subprocess
from pathlib import Path

print("Installing CUDA 12 runtime libraries...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "nvidia-cuda-runtime-cu12",
    "nvidia-cublas-cu12",
], check=True)

# Locate libraries installed for this notebook's Python, including user installs.
required_cuda_libs = ("libcudart.so.12", "libcublas.so.12")
site_roots = list(dict.fromkeys([*site.getsitepackages(), site.getusersitepackages()]))
cuda_lib_dirs = []
for base in site_roots:
    nvidia_dir = Path(base) / "nvidia"
    if nvidia_dir.is_dir():
        for lib_name in required_cuda_libs:
            for lib in sorted(nvidia_dir.rglob(lib_name)):
                if lib.is_file():
                    cuda_lib_dirs.append(str(lib.parent))
cuda_lib_dirs = list(dict.fromkeys(cuda_lib_dirs))

print("\nCUDA library directories:")
for directory in cuda_lib_dirs:
    print("  ", directory)

missing_cuda_libs = []
for lib_name in required_cuda_libs:
    found = any((Path(directory) / lib_name).is_file() for directory in cuda_lib_dirs)
    print(f"{'✅' if found else '❌'} {lib_name}")
    if not found:
        missing_cuda_libs.append(lib_name)
if missing_cuda_libs:
    raise RuntimeError("CUDA runtime installation is incomplete: " + ", ".join(missing_cuda_libs))

# Preserve existing driver/system paths; exclude empty entries and duplicates.
old_path = os.environ.get("LD_LIBRARY_PATH", "")
library_paths = cuda_lib_dirs + [str(WORKING)] + old_path.split(":")
os.environ["LD_LIBRARY_PATH"] = ":".join(dict.fromkeys(p for p in library_paths if p))

# A fresh process uses the updated loader path; file existence alone is insufficient.
cuda_check = subprocess.run(
    [sys.executable, "-c",
     "import ctypes; ctypes.CDLL('libcudart.so.12'); ctypes.CDLL('libcublas.so.12')"],
    capture_output=True, text=True, env=os.environ.copy(),
)
if cuda_check.returncode != 0:
    raise RuntimeError("CUDA runtime libraries could not be loaded:\n" + cuda_check.stderr)
print("\n✅ CUDA Runtime setup completed.")


## 📦 Cell 3 — Load current llama.cpp binaries

Copy binaries and their shared libraries from the attached `llama-cpp-bin` dataset into `/kaggle/working`, then verify the executable.

**Rebuild this dataset with current llama.cpp using `1b_setup_llama.ipynb` before the first run.** A successful `--version` check only verifies that the binary starts; model-format support is checked when the model loads.


In [ ]:
import shutil, json, os, subprocess
from pathlib import Path

# ── Copy binaries ─────────────────────────────────────────────────────────────
binaries = ["llama-server", "llama-cli", "llama-mtmd-cli"]
for name in binaries:
    src = LLAMA_BIN_DIR / name
    dst = WORKING / name
    assert src.exists(), (
        f"❌ Binary '{name}' not found in dataset.\n"
        f"   Re-run 1b_setup_llama.ipynb to rebuild the dataset."
    )
    shutil.copy2(src, dst)
    dst.chmod(0o755)

# ── Copy shared libraries (.so) ───────────────────────────────────────────────
# llama-server is dynamically linked — it needs its .so files at runtime.
# We copy them to /kaggle/working/ alongside the binaries.
so_count = 0
for so_file in LLAMA_BIN_DIR.glob("*.so*"):
    shutil.copy2(so_file, WORKING / so_file.name)
    so_count += 1

# ── Set LD_LIBRARY_PATH ───────────────────────────────────────────────────────
# Tell the OS where to find the .so files when llama-server starts.
ld_dirs = [*os.environ.get("LD_LIBRARY_PATH", "").split(":"), str(WORKING)]
ld_path = ":".join(dict.fromkeys(p for p in ld_dirs if p))
os.environ["LD_LIBRARY_PATH"] = ld_path

# ── Verify binary works ───────────────────────────────────────────────────────
test = subprocess.run(
    [str(LLAMA_SERVER), "--version"],
    capture_output=True, text=True,
    env={**os.environ, "LD_LIBRARY_PATH": ld_path}
)

if test.returncode != 0:
    print("❌ llama-server failed sanity check:")
    print(test.stderr[:500])
    raise RuntimeError("Binary verification failed — check dataset contents.")

# ── Print summary ─────────────────────────────────────────────────────────────
ver_file = LLAMA_BIN_DIR / "version.json"
if ver_file.exists():
    info = json.loads(ver_file.read_text())
    print("✅ llama.cpp binaries loaded.")
    print(f"   Commit  : {info.get('llama_cpp_commit')}  ({info.get('commit_date')})")
    print(f"   Version : {info.get('llama_server_ver')}")
    print(f"   CUDA    : {info.get('cuda_arch')}")
else:
    ver_str = (test.stdout or test.stderr).strip().splitlines()[0]
    print(f"✅ llama.cpp binaries loaded: {ver_str}")

print(f"   Copied  : {len(binaries)} binaries + {so_count} shared libraries")
print(f"   LD path : {ld_path}")

## ☁️ Cell 4 — Install Cloudflare Tunnel

`cloudflared` exposes the local API through a temporary HTTPS URL without a Cloudflare account. The URL changes for each session. Availability and connection behavior depend on the tunnel service; this does not guarantee uninterrupted requests.


In [ ]:
%%bash
wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
     -O /usr/local/bin/cloudflared
chmod +x /usr/local/bin/cloudflared
echo "✅ cloudflared installed: $(cloudflared --version)"

## 🧠 Cell 5 — Launch llama-server

Starts `llama-server` in the background, then waits until `/health` returns `{"status": "ok"}`
before continuing. Checking for `ok` (not just HTTP 200) ensures the model is fully loaded.

**Thinking mode** is not set here — it is passed per request by `chat.py`.
No server restart needed to switch modes.

In [ ]:
import subprocess, time, requests, os

# Kill any leftover llama-server from a previous run
subprocess.run(["pkill", "-f", "llama-server"], capture_output=True)
time.sleep(2)

# LD_LIBRARY_PATH must be in the environment of the subprocess too
env = os.environ.copy()  # Includes CUDA runtime, working directory and driver paths

cmd = [
    str(LLAMA_SERVER),
    "--model",            str(MODEL_PATH),
    "--alias",            MODEL_ALIAS,
    "--host",             "0.0.0.0",
    "--port",             str(PORT),
    "--n-gpu-layers",     str(N_GPU_LAYERS),
    "--ctx-size",         str(CTX_SIZE),
    "--split-mode",       "layer",  # distribute layers across the available GPUs
    "--parallel",         "1",      # one full context; predictable memory use
    "--jinja",                     # use the GGUF chat template
    "--temp",             str(TEMPERATURE),
    "--top-p",            str(TOP_P),
    "--top-k",            str(TOP_K),
    "--min-p",            str(MIN_P),
    "--presence-penalty", str(PRESENCE_PENALTY),
    "--log-file",         "/tmp/llama-server.log",
    # Thinking mode is NOT set here — passed per request by chat.py
]

if MMPROJ_PATH is not None:
    cmd += ["--mmproj", str(MMPROJ_PATH)]

print("🚀 Starting llama-server...")
print(f"   Model  : {MODEL_FILE}")
print(f"   mmproj : {MMPROJ_PATH or 'disabled (text only)'}")
print(f"   💡 Thinking: use chat.py --thinking  (no restart needed)")
print()

server_proc = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env=env)

print("⏳ Loading model (duration depends on input storage and GPU offloading)...")

# Wait for {"status": "ok"} — not just HTTP 200
# llama-server returns 200 + {"status":"loading"} while loading,
# and 200 + {"status":"ok"} only when truly ready to serve requests.
timeout, interval, elapsed, server_ok = 600, 15, 0, False
while elapsed < timeout:
    if server_proc.poll() is not None:
        print(f"   llama-server exited with code {server_proc.returncode}.")
        break
    try:
        r = requests.get(f"http://localhost:{PORT}/health", timeout=2)
        if r.status_code == 200 and r.json().get("status") == "ok":
            server_ok = True
            break
    except Exception:
        pass
    print(f"   [{elapsed:>3}s] Still loading...", flush=True)
    time.sleep(interval)
    elapsed += interval

if not server_ok:
    print("\n❌ Server failed to start. Last log lines:")
    try:
        with open("/tmp/llama-server.log") as f:
            print("".join(f.readlines()[-20:]))
    except Exception:
        print("   (log file not found — binary may have crashed before writing logs)")
    raise RuntimeError(
        "Server failed to become healthy. Check the log above; rebuild llama.cpp "
        "for unsupported GGUF/tensor errors, or reduce CTX_SIZE/N_GPU_LAYERS for OOM."
    )

print(f"\n✅ Server is ready! (started in {elapsed}s)")
print(f"   Local endpoint: http://localhost:{PORT}/v1")

## 🌐 Cell 6 — Start Cloudflare Tunnel & get public URL

In [ ]:
import re

tunnel_log = "/tmp/cloudflare.log"
tunnel_url = None

with open(tunnel_log, "w") as log:
    tunnel_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
        stdout=log, stderr=log,
    )

print("⏳ Waiting for Cloudflare tunnel URL...")

for _ in range(30):
    time.sleep(2)
    try:
        with open(tunnel_log) as f:
            match = re.search(r"https://[\w-]+\.trycloudflare\.com", f.read())
        if match:
            tunnel_url = match.group(0)
            break
    except Exception:
        pass

if not tunnel_url:
    print("❌ Could not get Cloudflare URL. Last log:")
    with open(tunnel_log) as f:
        print(f.read()[-1000:])
    raise RuntimeError("Cloudflare tunnel failed to start.")

print()
print("═" * 60)
print(f"  🔗 YOUR SERVER URL")
print(f"  {tunnel_url}")
print("═" * 60)
print()
print(f"  OpenAI endpoint : {tunnel_url}/v1")
print(f"  Direct answers  : python chat.py --url {tunnel_url} --model {MODEL_ALIAS}")
print(f"  Thinking mode   : python chat.py --url {tunnel_url} --model {MODEL_ALIAS} --thinking")

## 🔁 Cell 7 — Keep-alive loop

Monitors server health every 5 minutes while the Kaggle session is running. This loop does not extend Kaggle session limits.

**To stop:** interrupt the kernel (■ button) or let the session expire.

In [ ]:
import datetime

start_time     = datetime.datetime.now()
check_interval = 300

print(f"🟢 Server running since {start_time.strftime('%H:%M:%S')}")
print(f"   URL     : {tunnel_url}")
print(f"   Model   : {MODEL_FILE}")
print("   Session : subject to your current Kaggle runtime limits")
print()
print(f"   Connect : python chat.py --url {tunnel_url} --model {MODEL_ALIAS}")
print(f"   Thinking: python chat.py --url {tunnel_url} --model {MODEL_ALIAS} --thinking")
print()
print("   Press ■ (interrupt kernel) to stop.")
print("─" * 60)

try:
    while True:
        time.sleep(check_interval)
        elapsed = datetime.datetime.now() - start_time
        h, rem  = divmod(int(elapsed.total_seconds()), 3600)
        m       = rem // 60
        try:
            r      = requests.get(f"http://localhost:{PORT}/health", timeout=5)
            status = "✅ healthy" if r.status_code == 200 else f"⚠️ {r.status_code}"
        except Exception:
            status = "❌ unreachable"
        ts = datetime.datetime.now().strftime('%H:%M:%S')
        print(f"  [{ts}]  Uptime: {h}h {m:02d}m  |  {status}")
        if "❌" in status:
            print("\n⚠️  Server crashed. Check /tmp/llama-server.log")
            break
except KeyboardInterrupt:
    print("\n🛑 Shutting down...")
    server_proc.terminate()
    tunnel_proc.terminate()
    print("   Done.")